# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
ECOHOME_SYSTEM_PROMPT = """You are the EcoHome Energy Advisor, an assistant that helps customers with
solar panels, electric vehicles, smart thermostats and other smart-home devices decide when and how
to use energy in order to reduce their electricity costs and environmental impact.

Make data-driven decisions. Always use the available tools to obtain real data and base your answer
on it; never invent numbers.

Process for every request:
1. Determine what the customer needs (the device, the relevant time frame, and the cost, comfort or solar objective).
2. Retrieve the required data before drawing any conclusion:
   - get_weather_forecast for expected sunlight and solar potential
   - get_electricity_prices for time-of-use (peak vs off-peak) rates
   - query_energy_usage, query_solar_generation and get_recent_energy_summary for the customer's own history
3. Use search_energy_tips to retrieve a relevant best practice and cite the tip you used.
4. Analyze the data: when electricity is cheapest, when solar output is highest, and what the usage history indicates.
5. Use calculate_energy_savings to quantify the benefit; do not calculate savings manually.
6. Provide one clear recommendation.

Use the current date given in the context when querying historical usage or solar data.

Tool selection:
- For questions about the user's own past usage or solar output, you MUST call
  query_energy_usage, query_solar_generation or get_recent_energy_summary.
- For "how can I save / best practice" questions, you MUST call search_energy_tips.

Capabilities:
- analyze energy usage and solar generation
- recommend the best time to run a device (EV, HVAC, dishwasher, pool pump, appliances)
- combine weather forecasts and electricity prices to maximize solar self-consumption
- calculate potential savings and environmental impact

Every recommendation must include:
- the concrete action and the recommended time window
- the data and tools used, and any assumptions made
- the expected cost and savings in numbers (kWh, $, %) when the data is available
- a relevant citation from the knowledge base when applicable

Be specific and quantitative. If a tool fails or data is missing, state this clearly and provide a
careful best estimate rather than inventing precise values.

Example questions you should be able to answer:
- "When should I charge my electric car tomorrow to minimize cost and maximize solar power?"
- "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?"
- "Suggest three ways I can reduce energy use based on my usage history."
- "How much can I save by running my dishwasher during off-peak hours?"
- "What's the best time to run my pool pump this week based on the weather forecast?"
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [ ]:
%%sql


In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context=f"Location: San Francisco, CA. Today's date is {datetime.today().isoformat()}."
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric vehicle (EV) tomorrow (June 20, 2026), I recommend charging during the off-peak hours from **12:00 AM to 6:00 AM**. 

### Data Analysis:
1. **Electricity Prices**:
   - Off-Peak Rate: $0.082 per kWh (12:00 AM - 6:00 AM)
   - Mid-Peak Rate: $0.123 per kWh (6:00 AM - 4:00 PM)
   - Peak Rate: $0.2562 per kWh (4:00 PM - 9:00 PM)

2. **Solar Generation Forecast**:
   - Solar generation is expected to peak around **12:00 PM to 2:00 PM** with the highest output of approximately **900 W/m²**. However, charging during the night will allow you to take advantage of the lower electricity rates.

3. **Your EV Usage History**:
   - Your total EV consumption over the past 19 days is **3969.87 kWh**. Charging during off-peak hours can save you significantly on your electricity bill.

### Expected Savings:
- By charging your EV during the off-peak hours, you can save approximately **$325.53** based on your current usage and the off-

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices
- query_energy_usage
- query_solar_generation
- calculate_energy_savings


## 2. Define Test Cases

In [7]:
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "A time-window recommendation that uses cheap off-peak hours and/or the midday solar window, with cost/solar reasoning.",
    },
    {
        "id": "thermostat_price_spike",
        "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "A setpoint recommendation that raises the temperature (in summer) during the price peak and suggests pre-cooling, with reasoning.",
    },
    {
        "id": "dishwasher_offpeak_savings",
        "question": "How much can I save by running my dishwasher during off-peak hours instead of during the evening peak?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "A concrete savings estimate in $ and % comparing peak vs off-peak pricing.",
    },
    {
        "id": "pool_pump_schedule",
        "question": "What's the best time to run my pool pump this week based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "A schedule recommendation favoring sunny, low-price hours, with reasoning from the forecast.",
    },
    {
        "id": "reduce_usage_from_history",
        "question": "Suggest three ways I can reduce my energy use based on my recent usage history.",
        "expected_tools": ["get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "Three actionable tips grounded in the usage history and citing knowledge-base best practices.",
    },
    {
        "id": "solar_maximization",
        "question": "How can I maximize the use of my own solar power instead of exporting it to the grid?",
        "expected_tools": ["query_solar_generation", "search_energy_tips"],
        "expected_response": "Advice to shift flexible loads into the midday solar window, citing self-consumption tips.",
    },
    {
        "id": "ev_vs_appliances_cost",
        "question": "Last week, which used more energy and cost more: my EV or my appliances?",
        "expected_tools": ["query_energy_usage"],
        "expected_response": "A comparison of consumption (kWh) and cost ($) per device type from the usage data.",
    },
    {
        "id": "hvac_summer_optimization",
        "question": "How should I run my HVAC on a hot sunny day to cut cost without losing comfort?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "A pre-cooling strategy during solar/off-peak hours and a higher setpoint during the peak, with cited tips.",
    },
    {
        "id": "battery_peak_shaving",
        "question": "When should I charge and discharge my home battery to save the most money?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Charge during off-peak/solar and discharge during the evening peak (peak shaving), with reasoning.",
    },
    {
        "id": "weekly_savings_estimate",
        "question": "If I shift my EV charging from peak to off-peak, what are my estimated weekly and annual savings?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "A quantified savings estimate (weekly and annual $) based on the peak/off-peak price difference.",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [8]:
CONTEXT = f"Location: San Francisco, CA. Today's date is {datetime.today().isoformat()}."

In [9]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )

        # log the tools the agent actually called
        tools_used = [m.name for m in response.get("messages", []) if m.model_dump().get("tool_call_id")]

        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'tools_used': tools_used,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'tools_used': [],
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_price_spike
Question: What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?
--------------------------------------------------

Test 3: dishwasher_offpeak_savings
Question: How much can I save by running my dishwasher during off-peak hours instead of during the evening peak?
--------------------------------------------------

Test 4: pool_pump_schedule
Question: What's the best time to run my pool pump this week based on the weather forecast?
--------------------------------------------------

Test 5: reduce_usage_from_history
Question: Suggest three ways I can reduce my energy use based on my recent usage history.
--------------------------------------------------

Test 6: solar_maximization
Question: How can I maximize the us

In [10]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content="Location: San Francisco, CA. Today's date is 2026-06-19T12:35:52.932038.", additional_kwargs={}, response_metadata={}, id='ca71f1ae-69b5-4d3c-97f8-6845d895d73d'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='08e44ca3-7576-433f-ab73-efe038cbb40d'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 1433, 'total_tokens': 1562, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18',

## 4. Evaluate Responses

In [11]:
import os
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI


class _ResponseScore(BaseModel):
    """Judge scores for one response (each from 0.0 to 1.0)."""
    accuracy: float = Field(description="Are the facts, numbers and recommendations correct and plausible? 0.0-1.0")
    relevance: float = Field(description="Does it directly address the question? 0.0-1.0")
    completeness: float = Field(description="Does it cover the needed detail and an actionable recommendation? 0.0-1.0")
    usefulness: float = Field(description="Is it specific, actionable and genuinely helpful? 0.0-1.0")
    feedback: str = Field(description="A 2-3 sentence assessment of strengths and weaknesses")

In [12]:
# LLM as a judge
_judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY"),
)
_judge = _judge_llm.with_structured_output(_ResponseScore)

_METRICS = ["accuracy", "relevance", "completeness", "usefulness"]

def _to_unit_interval(value):
    """Force a value into the [0.0, 1.0] range (returns 0.0 if it is not a number)."""
    try:
        return max(0.0, min(1.0, float(value)))
    except (TypeError, ValueError):
        return 0.0

In [13]:
# Create a response evaluator
def evaluate_response(question, final_response, expected_response):
    """Evaluate one response with an LLM judge.

    Returns accuracy, relevance, completeness, usefulness (each 0.0-1.0), an
    overall score. If the judge call fails,
    the scores are returned as None so a single failure does not distort the
    aggregated report.
    """
    prompt = (
        "You are a strict evaluator for an energy-advisor assistant. "
        "Score the actual answer against the question and the expected answer. "
        "Each score is between 0.0 (poor) and 1.0 (excellent).\n\n"
        f"QUESTION:\n{question}\n\n"
        f"EXPECTED (what a good answer should contain):\n{expected_response}\n\n"
        f"ACTUAL ANSWER:\n{final_response}"
    )
    try:
        result = _judge.invoke(prompt)
        scores = {metric: _to_unit_interval(getattr(result, metric)) for metric in _METRICS}
        scores["overall"] = round(sum(scores.values()) / len(_METRICS), 3)
        scores["feedback"] = result.feedback
        return scores
    except Exception as e:
        failed = {metric: None for metric in _METRICS}
        failed["overall"] = None
        failed["feedback"] = f"Evaluation could not be completed: {e}"
        return failed

In [14]:
def _extract_used_tools(messages):
    """Return the tool names actually called in a LangGraph message list."""
    used = []
    if not isinstance(messages, list):
        return used
    for msg in messages:
        try:
            obj = msg.model_dump()
        except Exception:
            obj = msg if isinstance(msg, dict) else {}
        name = getattr(msg, "name", None) or obj.get("name")

        if obj.get("tool_call_id") and name:
            used.append(name)
    return used

In [15]:
# Create a tool usage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate whether the right tools were used.
    """
    used = _extract_used_tools(messages)
    used_set, expected_set = set(used), set(expected_tools or [])

    if used_set:
        appropriateness = len(used_set & expected_set) / len(used_set)
    else:
        appropriateness = 1.0 if not expected_set else 0.0
    completeness = len(used_set & expected_set) / len(expected_set) if expected_set else 1.0

    missing = sorted(expected_set - used_set)
    extra = sorted(used_set - expected_set)
    return {
        "tools_used": sorted(used_set),
        "expected_tools": sorted(expected_set),
        "tool_appropriateness": round(appropriateness, 3),
        "tool_completeness": round(completeness, 3),
        "tool_overall": round((appropriateness + completeness) / 2, 3),
        "missing_tools": missing,
        "extra_tools": extra,
        "feedback": (
            f"Used {sorted(used_set) or 'no'} tools; "
            f"missing {missing or 'none'}; extra {extra or 'none'}."
        ),
    }

In [16]:
# Generate a comprehensive evaluation report
def generate_evaluation_report(results=None):
    """Aggregate the response and tool evaluations across all test results."""
    if results is None:
        results = test_results

    per_test = []
    for r in results:
        resp = r.get("response")

        if isinstance(resp, dict) and resp.get("messages"):
            final_text = resp["messages"][-1].content
            messages = resp["messages"]
        else:
            final_text = str(resp)
            messages = []
        per_test.append({
            "test_id": r.get("test_id", "?"),
            "response_eval": evaluate_response(r["question"], final_text, r["expected_response"]),
            "tool_eval": evaluate_tool_usage(messages, r.get("expected_tools", [])),
        })

    def _avg(section, key):
        vals = [p[section].get(key) for p in per_test if isinstance(p[section].get(key), (int, float))]
        return round(sum(vals) / len(vals), 3) if vals else 0.0

    overall = {
        "accuracy": _avg("response_eval", "accuracy"),
        "relevance": _avg("response_eval", "relevance"),
        "completeness": _avg("response_eval", "completeness"),
        "usefulness": _avg("response_eval", "usefulness"),
        "response_overall": _avg("response_eval", "overall"),
        "tool_appropriateness": _avg("tool_eval", "tool_appropriateness"),
        "tool_completeness": _avg("tool_eval", "tool_completeness"),
    }
    strengths = [k for k, v in overall.items() if v >= 0.7]
    weaknesses = [k for k, v in overall.items() if v < 0.7]

    recommendations = []
    if overall["tool_completeness"] < 0.7:
        recommendations.append("Call all relevant tools (weather, prices, history) before answering.")
    if overall["completeness"] < 0.7:
        recommendations.append("Always include a concrete action, a time window and quantified savings.")
    if overall["accuracy"] < 0.7:
        recommendations.append("Base every number on tool data instead of estimates.")
    if not recommendations:
        recommendations.append("Performance is strong; extend coverage with more edge cases.")

    return {
        "num_tests": len(per_test),
        "overall_scores": overall,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
        "per_test": per_test,
    }


def display_evaluation_report(report):
    """Display the evaluation report in a readable, structured form."""
    print("=" * 60)
    print("ECOHOME ENERGY ADVISOR - EVALUATION REPORT")
    print("=" * 60)
    print(f"Tests evaluated: {report['num_tests']}\n")

    print("Overall scores:")
    for key, value in report["overall_scores"].items():
        print(f"  {key:<22} {value}")

    print("\nStrengths (>= 0.7):")
    for s in report["strengths"] or ["(none)"]:
        print(f"  + {s}")

    print("\nWeaknesses (< 0.7):")
    for w in report["weaknesses"] or ["(none)"]:
        print(f"  - {w}")

    print("\nRecommendations:")
    for rec in report["recommendations"]:
        print(f"  * {rec}")

    print("\nPer-test summary:")
    for p in report["per_test"]:
        rev, te = p["response_eval"], p["tool_eval"]
        print(f"  [{p['test_id']:<24}] response={rev.get('overall')}  tool_overall={te.get('tool_overall')}")
    print("=" * 60)

In [17]:
# run the evaluation across all test_results and display the report
report = generate_evaluation_report(test_results)
display_evaluation_report(report)

ECOHOME ENERGY ADVISOR - EVALUATION REPORT
Tests evaluated: 10

Overall scores:
  accuracy               0.97
  relevance              0.99
  completeness           0.95
  usefulness             0.96
  response_overall       0.968
  tool_appropriateness   0.798
  tool_completeness      0.85

Strengths (>= 0.7):
  + accuracy
  + relevance
  + completeness
  + usefulness
  + response_overall
  + tool_appropriateness
  + tool_completeness

Weaknesses (< 0.7):
  - (none)

Recommendations:
  * Performance is strong; extend coverage with more edge cases.

Per-test summary:
  [ev_charging_1           ] response=0.9  tool_overall=0.7
  [thermostat_price_spike  ] response=0.775  tool_overall=1.0
  [dishwasher_offpeak_savings] response=1.0  tool_overall=0.833
  [pool_pump_schedule      ] response=1.0  tool_overall=0.75
  [reduce_usage_from_history] response=1.0  tool_overall=1.0
  [solar_maximization      ] response=1.0  tool_overall=0.75
  [ev_vs_appliances_cost   ] response=1.0  tool_overall=1